# LLM Science BM25 RAG

Retrieve Wikipedia STEM paragraphs with BM25, then rank options using length, prompt TF-IDF, and context TF-IDF.


In [ ]:
import csv
import math
import re
from collections import Counter
from pathlib import Path

from datasets import load_from_disk

OPTIONS = ("A", "B", "C", "D", "E")
TFIDF_WEIGHT = 0.25
RAG_WEIGHT = 0.1
TOP_K = 10
WORD_RE = re.compile(r"[A-Za-z0-9]+(?:[-'][A-Za-z0-9]+)?")
STOPWORDS = {
    "the", "of", "and", "to", "in", "a", "is", "are", "which", "following",
    "statement", "statements", "accurately", "describes", "describe", "what", "how",
    "by", "for", "with", "as", "an", "on", "from", "that", "it", "this", "its",
    "be", "was", "were", "can", "into", "or", "not", "due", "between", "among",
    "about", "refers", "system", "systems", "object", "objects", "law", "theory",
    "mass", "data", "time", "answer", "option",
}


def find_csv(name):
    candidates = sorted(Path("/kaggle/input").glob(f"**/{name}"))
    if not candidates:
        return None
    return candidates[0]


def find_corpus_dir():
    known = Path("/kaggle/input/all-paraphs-parsed-expanded")
    if known.exists() and (known / "dataset_info.json").exists():
        return known
    for path in sorted(Path("/kaggle/input").rglob("dataset_info.json")):
        parent = path.parent
        if list(parent.glob("*.arrow")):
            return parent
    raise FileNotFoundError("No HuggingFace arrow corpus found under /kaggle/input")


def read_rows(path):
    with path.open(newline="", encoding="utf-8") as src:
        return list(csv.DictReader(src))


def tokens(text):
    return [word.lower() for word in WORD_RE.findall(text or "")]


def content_tokens(text):
    return [word for word in tokens(text) if len(word) >= 3 and word not in STOPWORDS]


def word_count(text):
    return len(tokens(text))


def zscores(values):
    mean = sum(values) / len(values)
    var = sum((value - mean) ** 2 for value in values) / len(values)
    scale = math.sqrt(var) or 1.0
    return [(value - mean) / scale for value in values]


def build_idf(rows):
    docs = []
    for row in rows:
        docs.append(set(content_tokens(row["prompt"])))
        for option in OPTIONS:
            docs.append(set(content_tokens(row[option])))
    df = Counter()
    for doc in docs:
        df.update(doc)
    n_docs = len(docs)
    return {term: math.log((n_docs + 1) / (count + 1)) + 1 for term, count in df.items()}


def tfidf_vector(words, idf):
    counts = Counter(words)
    return {word: count * idf.get(word, 1.0) for word, count in counts.items()}


def cosine(a, b):
    numerator = sum(value * b.get(key, 0.0) for key, value in a.items())
    norm_a = math.sqrt(sum(value * value for value in a.values()))
    norm_b = math.sqrt(sum(value * value for value in b.values()))
    if not norm_a or not norm_b:
        return 0.0
    return numerator / (norm_a * norm_b)


class BM25Retriever:
    def __init__(self, documents, k1=1.5, b=0.75):
        self.n_docs = len(documents)
        self.k1 = k1
        self.b = b
        self.doc_lens = [len(doc) for doc in documents]
        self.avgdl = sum(self.doc_lens) / self.n_docs if self.n_docs else 1.0
        self.term_freqs = [Counter(doc) for doc in documents]
        self.inverted_index = {}
        df = Counter()
        for idx, tf_counter in enumerate(self.term_freqs):
            for term, tf in tf_counter.items():
                df[term] += 1
                self.inverted_index.setdefault(term, []).append((idx, tf))
        self.idf = {
            term: math.log(1 + (self.n_docs - freq + 0.5) / (freq + 0.5))
            for term, freq in df.items()
        }

    def retrieve(self, query_tokens, top_k):
        scores = {}
        for term in set(query_tokens):
            if term not in self.idf:
                continue
            idf = self.idf[term]
            for idx, tf in self.inverted_index.get(term, []):
                dl = self.doc_lens[idx]
                denom = tf + self.k1 * (1 - self.b + self.b * dl / self.avgdl)
                scores[idx] = scores.get(idx, 0.0) + idf * (tf * (self.k1 + 1)) / denom
        if not scores:
            return list(range(min(top_k, self.n_docs)))
        return sorted(scores, key=lambda idx: (-scores[idx], idx))[:top_k]


def load_corpus(corpus_dir):
    dataset = load_from_disk(str(corpus_dir))
    texts = []
    tokenized = []
    for row in dataset:
        title = row.get("title") or ""
        section = row.get("section") or ""
        body = row.get("text") or ""
        text = " ".join(part for part in (title, section, body) if part).strip()
        if not text:
            continue
        texts.append(text)
        tokenized.append(content_tokens(text))
    return texts, tokenized


def rank_row(row, retriever, corpus_texts, exam_idf, context_cache):
    cache_key = row.get("id", row["prompt"])
    if cache_key not in context_cache:
        query = content_tokens(row["prompt"])
        top_indices = retriever.retrieve(query, TOP_K)
        context_cache[cache_key] = "\n".join(corpus_texts[idx] for idx in top_indices)

    context = context_cache[cache_key]
    prompt_vec = tfidf_vector(content_tokens(row["prompt"]), exam_idf)
    context_vec = tfidf_vector(content_tokens(context), exam_idf)

    length_values = [word_count(row[option]) for option in OPTIONS]
    prompt_tfidf_values = [
        cosine(prompt_vec, tfidf_vector(content_tokens(row[option]), exam_idf))
        for option in OPTIONS
    ]
    rag_values = [
        cosine(context_vec, tfidf_vector(content_tokens(row[option]), exam_idf))
        for option in OPTIONS
    ]

    scores = [
        length_z + TFIDF_WEIGHT * prompt_tfidf_z + RAG_WEIGHT * rag_z
        for length_z, prompt_tfidf_z, rag_z in zip(
            zscores(length_values), zscores(prompt_tfidf_values), zscores(rag_values)
        )
    ]
    return [
        option
        for _, option in sorted(zip(scores, OPTIONS), key=lambda item: (-item[0], item[1]))
    ]


train_path = find_csv("train.csv")
test_path = find_csv("test.csv")
corpus_dir = find_corpus_dir()
if test_path is None:
    raise FileNotFoundError("No test.csv found under /kaggle/input")

train_rows = read_rows(train_path) if train_path else []
test_rows = read_rows(test_path)
corpus_texts, corpus_tokens = load_corpus(corpus_dir)
retriever = BM25Retriever(corpus_tokens)
exam_idf = build_idf(train_rows + test_rows)
context_cache = {}

print(f"Using corpus: {corpus_dir}")
print(f"Corpus docs: {len(corpus_texts)}")
print(f"Train rows: {len(train_rows)} Test rows: {len(test_rows)}")

output_path = Path("/kaggle/working/submission.csv")
with output_path.open("w", newline="", encoding="utf-8") as dst:
    writer = csv.DictWriter(dst, fieldnames=["id", "prediction"])
    writer.writeheader()
    for row in test_rows:
        ranked = rank_row(row, retriever, corpus_texts, exam_idf, context_cache)
        writer.writerow({"id": row["id"], "prediction": " ".join(ranked[:3])})

print(f"Wrote {output_path}")
